# Week 48, Day 4: Probabilistic population codes

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w048/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Decoding a distribution

Write posterior(rs): compute log f_i(s) for every candidate s and neuron, form the log-posterior sum_i r_i log f_i(s), subtract its maximum, exponentiate and normalise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2006)
pref = np.linspace(-90, 90, 61); s_grid = np.linspace(-45, 45, 901); width = 15.0

def tuning(s, g):
    return g * np.exp(-(s - pref) ** 2 / (2 * width ** 2)) + 0.1

logF = -(s_grid[:, None] - pref[None]) ** 2 / (2 * width ** 2)     # log f_i(s) up to a constant

def posterior(rs):
    ...

def mean_sd(p):
    m = (p * s_grid).sum()
    return m, np.sqrt((p * (s_grid - m) ** 2).sum())

rs = r.poisson(tuning(5.0, 4))
p = posterior(rs)
print(f"posterior mean {mean_sd(p)[0]:.2f}, sd {mean_sd(p)[1]:.2f} (true stimulus 5)")
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4)); ax[0].bar(pref, rs, width=2.5); ax[0].set_xlabel("preferred stimulus")
ax[1].plot(s_grid, p); ax[1].axvline(5, color="k", ls=":"); ax[1].set_xlabel("stimulus"); plt.show()

In [ ]:
import numpy as np
def test_normalised():
    "the posterior sums to 1 and peaks near the true stimulus"
    p = posterior(r.poisson(tuning(5.0, 16)))
    assert abs(p.sum() - 1) < 1e-9 and abs(s_grid[np.argmax(p)] - 5.0) < 4

test_normalised()
print("All checks passed")

### Exercise: Width against gain

For gains 1, 4 and 16, decode 200 independent trials at s = 5 and average the posterior standard deviations.

In [ ]:
sd_by_gain = {}
for g in [1, 4, 16]:
    sd_by_gain[g] = ...
    print(f"gain {g:2d}: mean posterior sd {sd_by_gain[g]:.2f}")

In [ ]:
import numpy as np
def test_scaling():
    "multiplying the gain by 16 shrinks the average posterior sd by a factor between 3 and 5"
    assert 3 < sd_by_gain[1] / sd_by_gain[16] < 5

test_scaling()
print("All checks passed")

### Exercise: Adding spikes multiplies likelihoods

On 300 trials, draw spikes from a low-gain (3) and a high-gain (9) population at s = 5. Compare the posterior from r1 + r2 with the normalised product of the two separate posteriors, and record average widths.

In [ ]:
diffs, sds = [], []
for _ in range(300):
    r1, r2 = r.poisson(tuning(5.0, 3)), r.poisson(tuning(5.0, 9))
    p1, p2 = posterior(r1), posterior(r2)
    p_sum = ...
    p_prod = ...
    (m_a, s_a), (m_b, s_b) = mean_sd(p_sum), mean_sd(p_prod)
    diffs.append((abs(m_a - m_b), abs(s_a - s_b))); sds.append((mean_sd(p1)[1], mean_sd(p2)[1], s_a))
diffs, sds = np.array(diffs), np.array(sds)
max_mean_diff, max_sd_diff = diffs.max(0)
sd_one, sd_comb = sds[:, :2].mean(0), sds[:, 2].mean()
print(f"largest difference between summed-spike and product posteriors: mean {max_mean_diff:.1e}, sd {max_sd_diff:.1e}")
print(f"average sd: cue 1 {sd_one[0]:.2f}, cue 2 {sd_one[1]:.2f}, combined {sd_comb:.2f}, optimal {1 / np.sqrt(1 / sd_one[0] ** 2 + 1 / sd_one[1] ** 2):.2f}")

In [ ]:
import numpy as np
def test_exact():
    "the posterior from summed spikes equals the product of the two posteriors, on every trial"
    assert max_mean_diff < 1e-6 and max_sd_diff < 1e-6
def test_better():
    "the combined posterior is narrower than either single-cue posterior on average"
    assert sd_comb < min(sd_one)

test_exact()
test_better()
print("All checks passed")